# PSALM — Reproduce an evaluation number

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/PSALM/blob/main/notebooks/03_reproduce_eval.ipynb)

Re-run the official zero-shot read-off on a published checkpoint and reproduce a
number from the [results page](https://SharathSPhD.github.io/PSALM/results/). This
notebook clones the repository, pulls a checkpoint from the Hub, and runs the same
`scripts/official_eval.py` entry point used to produce the paper's tables. A GPU
runtime is recommended (Runtime → Change runtime type → GPU).

In [ ]:
!git clone --depth 1 https://github.com/SharathSPhD/PSALM.git
%cd PSALM
%pip install -q -e . sentencepiece
%pip install -q huggingface_hub

In [ ]:
import os
import shutil

from huggingface_hub import hf_hub_download

ARM = "a"  # a/b/c/d
REPO = f"qbz506/psalm-arm-{ARM}"
dst = f"data/checkpoints/strict_small/arm_{ARM.upper()}_seed_0"
os.makedirs(dst, exist_ok=True)
for fn in ["elc.pt"]:
    p = hf_hub_download(REPO, fn)
    shutil.copy(p, os.path.join(dst, fn))
# Tokenizer is also published in the model repo.
tok = hf_hub_download(REPO, "spm.model")
os.makedirs("data/tokenizer/strict_small", exist_ok=True)
shutil.copy(tok, "data/tokenizer/strict_small/spm.model")
print("checkpoint ready at", dst)

In [ ]:
# Run the official zero-shot suite (BLiMP + supplement + EWoK + entity tracking +
# WUG + COMPS) and print the Text Average. Use --skip-reading on CPU for speed.
!python scripts/official_eval.py \
    --ckpt data/checkpoints/strict_small/arm_A_seed_0/elc.pt \
    --name arm_A_seed_0

In [ ]:
import json
import pathlib

summ = pathlib.Path("data/checkpoints/strict_small/arm_A_seed_0/official_summary.json")
if summ.exists():
    print(json.dumps(json.loads(summ.read_text()), indent=2))
else:
    print("run the previous cell to produce official_summary.json")

The printed Text Average and per-task accuracies should match the published
results for the same checkpoint and seed. The (Super)GLUE average is produced
separately by `scripts/eval_finetune.py`, which fine-tunes a light classifier head
over the encoder for each task.